In [ ]:
!pip install -q langchain sentence-transformers faiss-cpu pypdf pandas accelerate bitsandbytes
!pip install -q langchain-community langchain-huggingface streamlit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d Cornell-University/arxiv
!unzip arxiv.zip

In [ ]:
%%writefile app.py
import streamlit as st
import json
import os
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# --- CONFIGURATION ---
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = "paste_your_token_here"

st.set_page_config(page_title="ArXiv CS Expert", layout="wide")

# --- DATA PROCESSING (STREAMING) ---
@st.cache_resource
def build_vector_db():
    json_path = 'arxiv-metadata-oai-snapshot.json'
    if not os.path.exists(json_path):
        st.error("Dataset not found! Please run the Kaggle download cell first.")
        return None

    papers = []
    with open(json_path, 'r') as f:
        for i, line in enumerate(f):
            item = json.loads(line)
            # Filter for Computer Science subset
            if 'cs.' in item.get('categories', ''):
                doc = f"Title: {item['title']}\nAbstract: {item['abstract']}"
                papers.append(doc)
            # Limit to 5000 papers for Colab performance/speed
            if len(papers) >= 5000:
                break

    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_db = FAISS.from_texts(papers, embeddings)
    return vector_db

# --- UI LAYOUT ---
st.title("🎓 ArXiv Computer Science Expert")
st.markdown("Retrieving insights from the latest scientific papers...")

vector_db = build_vector_db()

if vector_db and HF_TOKEN != "paste_your_token_here":
    # Initialize LLM (Mistral 7B)
    llm = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.3",
        huggingfacehub_api_token=HF_TOKEN,
        temperature=0.2,
        max_new_tokens=512
    )

    # Initialize Memory for Follow-up Questions
    if "memory" not in st.session_state:
        st.session_state.memory = ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True,
            output_key='answer'
        )

    # Setup Retrieval Chain
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        memory=st.session_state.memory,
        return_source_documents=True
    )

    # Chat Interface
    if "messages" not in st.session_state:
        st.session_state.messages = []

    for msg in st.session_state.messages:
        st.chat_message(msg["role"]).write(msg["content"])

    if prompt := st.chat_input("Ask a complex CS question..."):
        st.session_state.messages.append({"role": "user", "content": prompt})
        st.chat_message("user").write(prompt)

        with st.chat_message("assistant"):
            with st.spinner("Analyzing papers..."):
                response = qa_chain.invoke({"question": prompt})
                answer = response['answer']
                st.write(answer)

                # Show Sources
                with st.expander("See Research Sources"):
                    for doc in response['source_documents']:
                        st.caption(doc.page_content)

                st.session_state.messages.append({"role": "assistant", "content": answer})
else:
    if HF_TOKEN == "paste_your_token_here":
        st.warning("Please enter your Hugging Face Token in the code to begin.")

In [ ]:
print("\nYour Tunnel Password is:")
!curl ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501